In [1]:
!pip -q install "transformers>=4.44.2" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.1" datasets pillow pandas torch torchvision --upgrade

!pip install git+https://github.com/huggingface/transformers

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-ff7kmf6y
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-ff7kmf6y
  Resolved https://github.com/huggingface/transformers to commit 77e8b9f8dfc8e736ad2f603a5b2ae2b1076ed271
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# ============================================================
# GPU 메모리 상태 확인
# ============================================================
import torch

print("🔍 현재 GPU 메모리 상태:")
print(f"  할당됨: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"  예약됨: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"  최대 사용: {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GB")
print(f"  전체 용량: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
print(f"  여유 공간: {free_memory / 1024**3:.2f} GB")

if torch.cuda.memory_allocated(0) / 1024**3 > 35:
    print("\n⚠️ 경고: GPU 메모리가 거의 가득 찼습니다!")
    print("   이것이 느린 이유일 수 있습니다.")


🔍 현재 GPU 메모리 상태:
  할당됨: 0.00 GB
  예약됨: 0.00 GB
  최대 사용: 0.00 GB
  전체 용량: 39.49 GB
  여유 공간: 39.49 GB


# 셀 1: 라이브러리 임포트 및 기본 설정

In [3]:
import os, math, random
from datetime import datetime
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, get_linear_schedule_with_warmup

# 성능 최적화 설정
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cudnn.benchmark = True

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


# 셀 2: 학습 설정 (여기만 수정)

In [4]:
# 모델 설정
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
IMAGE_SIZE = 224  # 최종 이미지 크기
SEED = 42

# 학습 하이퍼파라미터
BATCH_SIZE = 4   # 안정적인 크기
GRAD_ACCUM = 4   # Effective batch = 16
EPOCHS = 1
LEARNING_RATE = 1e-4
NUM_WORKERS = 4  # 멀티프로세싱 활성화 (중요!)
NUM_TRAIN_SAMPLES = 3887  # 학습에 사용할 샘플 수 (None = 전체 사용)

# 버전 관리: 자동으로 날짜가 붙습니다
# 예: Qwen3-VL-4B-finetuned_20241026_143022
VERSION_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_SAVE_DIR = "/home/team036/content"
MODEL_SAVE_DIR = f"{BASE_SAVE_DIR}/Qwen3-VL-4B-finetuned_{VERSION_NAME}"

# 데이터 경로
TRAIN_CSV = "/home/team036/aichallenge/train.csv"
TEST_CSV = "/home/team036/aichallenge/test.csv"
NUM_TRAIN_SAMPLES = 3887  # None으로 설정하면 전체 사용

print(f"\n{'='*60}")
print(f"📦 Model Version: {VERSION_NAME}")
print(f"💾 Save Directory: {MODEL_SAVE_DIR}")
print(f"{'='*60}\n")

# 시드 설정
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


📦 Model Version: 20251026_044259
💾 Save Directory: /home/team036/content/Qwen3-VL-4B-finetuned_20251026_044259



# 셀 3: 데이터 로드 및 분할

In [5]:
print("Loading data...")
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# 학습데이터 샘플링
if NUM_TRAIN_SAMPLES is not None:
    train_df = train_df.sample(n=NUM_TRAIN_SAMPLES, random_state=SEED).reset_index(drop=True)
    print(f"✓ Sampled {NUM_TRAIN_SAMPLES} samples for training")
else:
    print(f"✓ Using all {len(train_df)} samples for training")

# 검증 데이터 분리 (85:15)
split_idx = int(len(train_df) * 0.85)
train_subset = train_df.iloc[:split_idx].reset_index(drop=True)
valid_subset = train_df.iloc[split_idx:].reset_index(drop=True)

print(f"✓ Train samples: {len(train_subset)}")
print(f"✓ Valid samples: {len(valid_subset)}")
print(f"✓ Test samples: {len(test_df)}")

Loading data...
✓ Sampled 3887 samples for training
✓ Train samples: 3303
✓ Valid samples: 584
✓ Test samples: 3887


# 셀 4: 프롬프트 생성 함수

In [6]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Output only the correct answer as a single lowercase letter."
    )

# 셀 5: 모델 및 프로세서 로드

In [7]:
print("\nLoading model and processor...")

# 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 프로세서 로드
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 베이스 모델 로드
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델 준비
base_model = prepare_model_for_kbit_training(base_model)
# base_model.gradient_checkpointing_enable()  # 메모리 충분하면 끄기 (1.5~2배 빠름)

print("✓ Base model loaded")


Loading model and processor...


Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.14s/it]


✓ Base model loaded


# 셀 6: LoRA 설정 및 PEFT 모델 생성

In [8]:
# LoRA 설정
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# LoRA 적용 확인
print("\n🔍 LoRA 적용 확인:")
lora_modules = [name for name, _ in model.named_modules() if 'lora' in name.lower()]
if lora_modules:
    print(f"✅ LoRA가 적용된 모듈 수: {len(lora_modules)}")
    print(f"   예시: {lora_modules[0]}")
else:
    print("⚠️  경고: LoRA가 적용되지 않았습니다!")
    print("   target_modules를 확인해야 합니다.")
    print("\n사용 가능한 modules 확인:")
    available_modules = set()
    for name, module in base_model.named_modules():
        if hasattr(module, 'weight') and len(list(module.parameters())) > 0:
            module_name = name.split('.')[-1]
            if 'proj' in module_name or 'mlp' in module_name or 'attn' in module_name:
                available_modules.add(module_name)
    print(f"   추천 target_modules: {sorted(available_modules)}")


trainable params: 16,515,072 || all params: 4,454,330,880 || trainable%: 0.3708

🔍 LoRA 적용 확인:
✅ LoRA가 적용된 모듈 수: 2268
   예시: base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_dropout


# 셀 7: Dataset 및 Collator 정의

In [9]:
class VQAMCDataset(Dataset):
    """최적화된 이미지 전처리"""
    def __init__(self, df, image_size=224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        
        # 바로 정사각형으로 리사이즈 (BILINEAR - 빠름)
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        
        user_text = build_mc_prompt(
            row["question"], row["a"], row["b"], row["c"], row["d"]
        )
        answer = str(row["answer"]).strip().lower()
        
        return {
            "image": img,
            "user_text": user_text,
            "answer": answer
        }

class VQACollator:
    def __init__(self, processor, system_instruct):
        self.processor = processor
        self.system_instruct = system_instruct
    
    def __call__(self, batch):
        images = []
        texts = []
        
        for sample in batch:
            img = sample["image"]
            user_text = sample["user_text"]
            answer = sample["answer"]
            
            messages = [
                {"role": "system", "content": [{"type": "text", "text": self.system_instruct}]},
                {"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": user_text}
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": answer}]}
            ]
            
            text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            
            images.append(img)
            texts.append(text)
        
        encoded = self.processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True
        )
        
        encoded["labels"] = encoded["input_ids"].clone()
        return encoded

print("✓ Dataset and Collator defined")

✓ Dataset and Collator defined


# 셀 8: DataLoader 생성

In [10]:
train_dataset = VQAMCDataset(train_subset, image_size=IMAGE_SIZE)
valid_dataset = VQAMCDataset(valid_subset, image_size=IMAGE_SIZE)

collate_fn = VQACollator(processor, SYSTEM_INSTRUCT)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    prefetch_factor=4,  # 미리 4개 배치 준비
    persistent_workers=True  # Worker 재사용
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True
)

print("✓ DataLoaders created")

✓ DataLoaders created


# 셀 9: Optimizer 및 Scheduler 설정

In [11]:
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# 올바른 스케줄러 step 계산
update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * update_steps_per_epoch
warmup_steps = int(num_training_steps * 0.03)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, num_training_steps)

# bf16은 GradScaler 불필요 (오버헤드 제거)
scaler = torch.cuda.amp.GradScaler(enabled=False)

print(f"\n{'='*60}")
print(f"Training Configuration")
print(f"  • Epochs: {EPOCHS}")
print(f"  • Batch size: {BATCH_SIZE}")
print(f"  • Gradient accumulation: {GRAD_ACCUM}")
print(f"  • Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  • Learning rate: {LEARNING_RATE}")
print(f"  • Update steps per epoch: {update_steps_per_epoch}")
print(f"  • Total training steps: {num_training_steps}")
print(f"  • Warmup steps: {warmup_steps}")
print(f"  • NUM_WORKERS: {NUM_WORKERS}")
print(f"  • Gradient checkpointing: OFF (faster)")
print(f"{'='*60}\n")



Training Configuration
  • Epochs: 1
  • Batch size: 4
  • Gradient accumulation: 4
  • Effective batch size: 16
  • Learning rate: 0.0001
  • Update steps per epoch: 207
  • Total training steps: 207
  • Warmup steps: 6
  • NUM_WORKERS: 4
  • Gradient checkpointing: OFF (faster)



/tmp/ipykernel_4037/3042354267.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=False)


# 디버깅: 병목 지점 찾기

In [11]:
import time

print("\n🔍 병목 지점 테스트...")

# 1. 데이터 로딩 속도
start = time.time()
batch = next(iter(train_loader))
data_time = time.time() - start
print(f"  데이터 로딩: {data_time:.2f}초")

# 2. GPU 전송 속도
start = time.time()
batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
gpu_time = time.time() - start
print(f"  GPU 전송: {gpu_time:.2f}초")

# 3. Forward pass 속도
start = time.time()
with torch.amp.autocast('cuda', dtype=torch.bfloat16):
    outputs = model(**batch)
forward_time = time.time() - start
print(f"  Forward pass: {forward_time:.2f}초")

# 4. Backward pass 속도
start = time.time()
loss = outputs.loss / GRAD_ACCUM
scaler.scale(loss).backward()
backward_time = time.time() - start
print(f"  Backward pass: {backward_time:.2f}초")

print(f"\n  총 시간: {data_time + gpu_time + forward_time + backward_time:.2f}초")
print(f"  병목: ", end="")
times = {"데이터 로딩": data_time, "GPU 전송": gpu_time, 
         "Forward": forward_time, "Backward": backward_time}
print(max(times, key=times.get))

optimizer.zero_grad(set_to_none=True)
print()


🔍 병목 지점 테스트...
  데이터 로딩: 0.72초
  GPU 전송: 0.00초


OutOfMemoryError: CUDA out of memory. Tried to allocate 80.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 17.44 MiB is free. Including non-PyTorch memory, this process has 39.46 GiB memory in use. Of the allocated memory 38.02 GiB is allocated by PyTorch, and 964.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# 셀 9.5: GPU 메모리 정리

In [14]:
import gc

print("🧹 GPU 메모리 정리 중...")

# 1. Python 가비지 컬렉션
gc.collect()

# 2. CUDA 캐시 정리
torch.cuda.empty_cache()

# 3. 메모리 상태 확인
print(f"GPU 메모리 상태:")
print(f"  할당됨: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"  예약됨: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"  최대 사용: {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GB")

# 4. 메모리 초기화
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()

print("✓ 메모리 정리 완료\n")

🧹 GPU 메모리 정리 중...
GPU 메모리 상태:
  할당됨: 3.89 GB
  예약됨: 4.13 GB
  최대 사용: 6.71 GB
✓ 메모리 정리 완료



# 셀 10: 학습 루프 실행

In [13]:
global_step = 0
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # ============== Training ==============
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        
        # bf16 autocast (scaler 없이)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM
        
        loss.backward()  # scaler 제거
        train_loss += loss.item()
        
        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            
            avg_loss = train_loss / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            train_loss = 0.0
    
    # 마지막 배치 처리
    if step % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
    
    # ============== Validation ==============
    model.eval()
    val_loss = 0.0
    val_steps = 0
    
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for batch in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss += outputs.loss.item()
            val_steps += 1
    
    avg_val_loss = val_loss / val_steps
    print(f"\n[Epoch {epoch+1}] Validation Loss: {avg_val_loss:.4f}")
    
    # ============== Best 모델만 저장 ==============
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"  ✓ New best model! Saving to {MODEL_SAVE_DIR}")
        
        # 디렉토리 생성
        os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
        
        # LoRA 어댑터 저장
        model.save_pretrained(MODEL_SAVE_DIR)
        processor.save_pretrained(MODEL_SAVE_DIR)
        
        # 학습 정보 저장
        with open(f"{MODEL_SAVE_DIR}/training_info.txt", "w") as f:
            f.write(f"Model Version: {VERSION_NAME}\n")
            f.write(f"Training Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Base Model: {MODEL_ID}\n")
            f.write(f"Epoch: {epoch+1}/{EPOCHS}\n")
            f.write(f"Validation Loss: {avg_val_loss:.4f}\n")
            f.write(f"Batch Size: {BATCH_SIZE}\n")
            f.write(f"Gradient Accumulation: {GRAD_ACCUM}\n")
            f.write(f"Learning Rate: {LEARNING_RATE}\n")
            f.write(f"Train Samples: {len(train_subset)}\n")
            f.write(f"Valid Samples: {len(valid_subset)}\n")

print(f"\n{'='*60}")
print(f"🎉 Training Complete!")
print(f"  • Best Validation Loss: {best_val_loss:.4f}")
print(f"  • Model Version: {VERSION_NAME}")
print(f"  • Saved at: {MODEL_SAVE_DIR}")
print(f"{'='*60}\n")

# GPU 메모리 정리
del model, optimizer, scheduler
torch.cuda.empty_cache()
print("✓ GPU memory cleared")

Epoch 1/1 [Train]:   0%|          | 0/826 [00:00<?, ?it/s]

/home/team036/.local/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/team036/.local/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Epoch 1/1 [Valid]: 100%|██████████| 146/146 [19:09<00:00,  7.87s/it]



[Epoch 1] Validation Loss: 2.5596
  ✓ New best model! Saving to /home/team036/content/Qwen3-VL-4B-finetuned_20251026_044259

🎉 Training Complete!
  • Best Validation Loss: 2.5596
  • Model Version: 20251026_044259
  • Saved at: /home/team036/content/Qwen3-VL-4B-finetuned_20251026_044259

✓ GPU memory cleared
